# Train the Frustum-PointNet box head on Kaggle GPU (with augmentation)

Trains the learned 3D-box head from **nocturnalnoob/lidar-rgb-frustum-fusion** on the full
KITTI set, adding the augmentation that the repo's honest benchmark showed was missing:

- **2D-box jitter** — randomly translate/scale the GT box before cropping the frustum, so the
  model sees *detector-like* boxes, not just pixel-perfect GT boxes (this is the fix for the
  train/deploy gap where the learned head scored mAP 0.006 on YOLO boxes vs 0.20 geometric).
- **Point dropout + jitter noise** — robustness to LiDAR sparsity at range.

**Before running:** in the notebook settings (right panel) set **Accelerator = GPU T4 x2**
(universally supported — avoids `no kernel image` errors), turn **Internet = On**, and
**Add Data → the KITTI 3D Object Detection dataset** (needs at least `velodyne/`, `calib/`,
`label_2/`; add `image_2/` too if you want the deployment eval in the last cell). Then
*Run All*. The trained `frustum_pointnet.pt` lands in `/kaggle/working/models/` for download.

> **Do not `pip install` anything that upgrades `torch`** in this notebook — reshuffling torch
> breaks Kaggle's GPU-matched build and causes `CUDA error: no kernel image is available`.
> Training needs only the preinstalled torch/numpy/scikit-learn/opencv. The optional eval cell
> installs ultralytics with `--no-deps` precisely so it can't touch torch.

In [ ]:
# 1. Environment + repo  (NB: nothing here changes torch — that would break CUDA on Kaggle)
import torch, sys, os, subprocess
print('torch', torch.__version__, '| built for CUDA', torch.version.cuda)
assert torch.cuda.is_available(), 'No GPU: right panel -> Accelerator = GPU (T4 x2).'
print('GPU:', torch.cuda.get_device_name(0), '| compute capability', torch.cuda.get_device_capability())
try:
    _ = torch.randn(64, device='cuda') @ torch.randn(64, 64, device='cuda'); torch.cuda.synchronize()
    print('CUDA kernel smoke test: OK')
except RuntimeError as e:
    raise SystemExit(
        'CUDA/torch mismatch ("no kernel image is available"). The active torch has no kernels '
        'for this GPU -- usually because torch was reinstalled/upgraded, or the GPU arch is '
        'unsupported. Fix: Run -> Factory reset the session, pick the "GPU T4 x2" accelerator, '
        'and do NOT pip-install anything that changes torch. Original error: ' + str(e))

REPO = '/kaggle/working/lidar-rgb-frustum-fusion'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/nocturnalnoob/lidar-rgb-frustum-fusion.git', REPO], check=True)
%cd $REPO
sys.path.insert(0, REPO)
print('repo ready; training needs only preinstalled torch/numpy/scikit-learn/opencv.')

In [ ]:
# 2. Locate the attached KITTI dataset and expose it where the repo's loaders expect it
import glob
hits = glob.glob('/kaggle/input/**/training/velodyne', recursive=True)
assert hits, ('No KITTI dataset found. Add Data -> a KITTI 3D Object Detection dataset '
              'that contains training/velodyne, training/calib, training/label_2.')
DATA_ROOT = os.path.dirname(os.path.dirname(hits[0]))   # dir that contains training/
print('KITTI root:', DATA_ROOT)
for sub in ['velodyne', 'calib', 'label_2', 'image_2']:
    p = os.path.join(DATA_ROOT, 'training', sub)
    print(f'  {sub:9s}: {"OK" if os.path.isdir(p) else "MISSING"}'
          f'{"  ("+str(len(os.listdir(p)))+" files)" if os.path.isdir(p) else ""}')

In [ ]:
# 3. Imports from the repo + config
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from src.data.kitti_loader import KittiLoader
from src.calibration.project_lidar import project_velo_to_image_indexed, rect_to_velo
from src.fusion.frustum_pointnet import (
    FrustumPointNet, SIZE_ANCHOR, CLASS_IDS, NUM_POINTS,
    normalize_frustum, sample_points, decode_prediction, _rotz)

CLASSES   = ('Car', 'Pedestrian', 'Cyclist')
MIN_PTS   = 30           # min points in a (jittered) frustum
EXPAND    = 0.30         # enlarge the GT crop so jittered boxes stay inside
EPOCHS    = 80
BATCH     = 64
LR        = 1e-3
# augmentation strength
JITTER_TRANS = 0.10      # box centre shift, fraction of box w/h
JITTER_SCALE = 0.12      # box scale +/- fraction
PT_KEEP_MIN  = 0.5       # random point-dropout keeps 50-100%
PT_NOISE     = 0.02      # metres of Gaussian jitter on points

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
loader = KittiLoader(DATA_ROOT)
frames = loader.list_frames()
val_frames   = [f for f in frames if f % 7 == 0]     # ~14% held out, frame-level
train_frames = [f for f in frames if f % 7 != 0]
print(f'{len(frames)} frames -> train {len(train_frames)} / val {len(val_frames)}')

In [ ]:
# 4. Extract frustum samples (enlarged crop + projected uv, so we can re-crop per jitter)
def build_samples(frames):
    out = []
    for n, idx in enumerate(frames):
        calib = loader.get_calib(idx); lidar = loader.get_lidar(idx)
        uv, depth, front = project_velo_to_image_indexed(lidar, calib)
        for o in loader.get_labels(idx):
            if o['type'] not in CLASSES or o['dimensions'][0] <= 0:
                continue
            x1, y1, x2, y2 = o['bbox']; w, h = x2 - x1, y2 - y1
            ex = [x1 - EXPAND * w, y1 - EXPAND * h, x2 + EXPAND * w, y2 + EXPAND * h]
            m = (front & (uv[:, 0] >= ex[0]) & (uv[:, 0] <= ex[2])
                 & (uv[:, 1] >= ex[1]) & (uv[:, 1] <= ex[3]) & (depth < 70))
            if int(m.sum()) < MIN_PTS:
                continue
            bottom = rect_to_velo(o['location'], calib)[0]
            hgt = o['dimensions'][0]
            out.append({
                'calib': calib,
                'pts': lidar[m][:, :3].astype(np.float32),
                'uv':  uv[m].astype(np.float32),
                'bbox': np.array(o['bbox'], dtype=np.float32),
                'cls': o['type'],
                'center': (bottom + np.array([0, 0, hgt / 2.0])).astype(np.float32),
                'size': o['dimensions'].astype(np.float32),     # h, w, l
                'yaw': np.float32(-o['rotation_y'] - np.pi / 2),
            })
        if (n + 1) % 500 == 0:
            print(f'  {n+1}/{len(frames)} frames, {len(out)} objects')
    return out

train_samples = build_samples(train_frames)
val_samples   = build_samples(val_frames)
print('objects  train:', len(train_samples), ' val:', len(val_samples))
for c in CLASSES:
    print(f'   {c}: train {sum(s["cls"]==c for s in train_samples)} '
          f'val {sum(s["cls"]==c for s in val_samples)}')

In [ ]:
# 5. Augmentation + Dataset
def jitter_bbox(bbox, rng):
    x1, y1, x2, y2 = bbox; w, h = x2 - x1, y2 - y1
    cx = (x1 + x2) / 2 + rng.uniform(-JITTER_TRANS, JITTER_TRANS) * w
    cy = (y1 + y2) / 2 + rng.uniform(-JITTER_TRANS, JITTER_TRANS) * h
    nw = w * (1 + rng.uniform(-JITTER_SCALE, JITTER_SCALE))
    nh = h * (1 + rng.uniform(-JITTER_SCALE, JITTER_SCALE))
    return np.array([cx - nw / 2, cy - nh / 2, cx + nw / 2, cy + nh / 2], dtype=np.float32)

def point_aug(pts, rng):
    keep = rng.uniform(PT_KEEP_MIN, 1.0)
    if pts.shape[0] * keep >= MIN_PTS:
        sel = rng.random(pts.shape[0]) < keep
        if sel.sum() >= MIN_PTS:
            pts = pts[sel]
    return pts + rng.normal(0, PT_NOISE, pts.shape).astype(np.float32)

class FrustumDS(Dataset):
    def __init__(self, samples, train):
        self.s = samples; self.train = train
    def __len__(self):
        return len(self.s)
    def __getitem__(self, i):
        s = self.s[i]
        rng = np.random.default_rng((i + 1) * (2 if self.train else 1))
        bbox = jitter_bbox(s['bbox'], rng) if self.train else s['bbox']
        u, v = s['uv'][:, 0], s['uv'][:, 1]
        m = (u >= bbox[0]) & (u <= bbox[2]) & (v >= bbox[1]) & (v <= bbox[3])
        pts = s['pts'][m]
        if pts.shape[0] < MIN_PTS:
            pts = s['pts']
        if self.train:
            pts = point_aug(pts, rng)
        pts_norm, theta, mean = normalize_frustum(pts, bbox, s['calib'])
        pts_s = sample_points(pts_norm, NUM_POINTS, rng).astype(np.float32)
        center = (_rotz(-theta) @ s['center'] - mean).astype(np.float32)
        yawc = s['yaw'] - theta
        oh = np.zeros(3, np.float32); oh[CLASS_IDS[s['cls']]] = 1
        head = np.array([np.sin(yawc), np.cos(yawc)], np.float32)
        size_resid = (s['size'] - SIZE_ANCHOR[s['cls']]).astype(np.float32)
        return pts_s, oh, center, size_resid, head

train_dl = DataLoader(FrustumDS(train_samples, True),  batch_size=BATCH, shuffle=True,  num_workers=2, drop_last=True)
val_dl   = DataLoader(FrustumDS(val_samples,   False), batch_size=BATCH, shuffle=False, num_workers=2)

In [ ]:
# 6. Train on GPU
model = FrustumPointNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
sl1 = nn.SmoothL1Loss()

def run_epoch(dl, train):
    model.train(train)
    tot = 0.0; nb = 0
    torch.set_grad_enabled(train)
    for pts, oh, c, sr, hd in dl:
        pts, oh = pts.to(device), oh.to(device)
        c, sr, hd = c.to(device), sr.to(device), hd.to(device)
        pc, ps, ph = model(pts, oh)
        ph = ph / ph.norm(dim=1, keepdim=True).clamp_min(1e-6)
        loss = sl1(pc, c) + sl1(ps, sr) + sl1(ph, hd)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    torch.set_grad_enabled(True)
    return tot / max(nb, 1)

os.makedirs('/kaggle/working/models', exist_ok=True)
CKPT = '/kaggle/working/models/frustum_pointnet.pt'
best = 1e9
for ep in range(EPOCHS):
    tr = run_epoch(train_dl, True)
    sched.step()
    if (ep + 1) % 5 == 0 or ep == 0:
        va = run_epoch(val_dl, False)
        if va < best:
            best = va
            torch.save(model.state_dict(), CKPT)
        print(f'epoch {ep+1:3d}/{EPOCHS}  train {tr:.4f}  val {va:.4f}  (best {best:.4f})')
print('saved best ->', CKPT)

In [ ]:
# 7. Term-by-term eval on val GT boxes (no jitter): learned head quality
model.load_state_dict(torch.load(CKPT, map_location=device)); model.eval()
def angle_err(a, b):
    d = abs((a - b + np.pi) % (2 * np.pi) - np.pi); return min(d, abs(np.pi - d))
ce, ye, se = [], [], []
with torch.no_grad():
    for s in val_samples:
        pts_norm, theta, mean = normalize_frustum(s['pts'], s['bbox'], s['calib'])
        pts = torch.tensor(sample_points(pts_norm, NUM_POINTS)[None]).float().to(device)
        oh = torch.zeros(1, 3, device=device); oh[0, CLASS_IDS[s['cls']]] = 1
        pc, ps, ph = model(pts, oh)
        ph = (ph / ph.norm(dim=1, keepdim=True).clamp_min(1e-6))[0].cpu().numpy()
        box = decode_prediction(pc[0].cpu().numpy(), ps[0].cpu().numpy(), ph, s['cls'], theta, mean)
        ce.append(np.hypot(*(box['center_velo'][:2] - s['center'][:2])))
        ye.append(angle_err(box['yaw_velo'], float(s['yaw'])))
        se.append(float(np.mean(box['size_hwl'] / s['size'])))
print(f'val objects: {len(val_samples)}')
print(f'  median center err: {np.median(ce):.3f} m   (repo baseline w/o aug: ~1.09 m)')
print(f'  median heading err: {np.median(ye):.3f} rad')
print(f'  median size ratio : {np.median(se):.3f}')

In [ ]:
# 8. (Optional, torch-safe) DEPLOYMENT eval on detector (YOLO) boxes: did the augmentation
#    close the gap? Needs image_2 attached. ultralytics is installed with --no-deps so it CANNOT
#    change torch (which would re-trigger the CUDA 'no kernel image' error). If its import still
#    fails, skip this on Kaggle and run the eval locally after downloading the weights.
import shutil, torch
shutil.copy(CKPT, os.path.join(REPO, 'models', 'frustum_pointnet.pt'))
subprocess.run(['pip', 'install', '-q', '--no-deps',
                'ultralytics', 'ultralytics-thop', 'py-cpuinfo'], check=False)
assert torch.cuda.is_available(), 'torch CUDA broke -- Factory reset and re-run without changing torch.'
try:
    import ultralytics  # noqa: F401
    print('=== LEARNED head (augmented), 300 frames ===')
    !python scripts/evaluate_dataset.py --data_dir "$DATA_ROOT" --head learned --limit 300 --conf 0.2
    print('=== GEOMETRIC head, 300 frames (reference) ===')
    !python scripts/evaluate_dataset.py --data_dir "$DATA_ROOT" --head geometric --limit 300 --conf 0.2
except ImportError as e:
    print('ultralytics unavailable here:', e)
    print('Skip on Kaggle; download the .pt and run locally:')
    print('  python scripts/evaluate_dataset.py --data_dir data/kitti --head learned')

## Use the trained head locally

1. **Download** `/kaggle/working/models/frustum_pointnet.pt` (Output panel → download), or
   *Save Version* to persist it as a Kaggle Dataset.
2. Drop it into your repo at `models/frustum_pointnet.pt` (overwriting the CPU-trained one).
3. Run inference on CPU as before:
   ```bash
   python scripts/run_fusion.py --idx 2 --head learned
   python scripts/evaluate_dataset.py --data_dir data/kitti --head learned   # check the gap
   ```
4. In the web app, pick the **Learned** 3D-head toggle.

If the deployment mAP in cell 8 is now meaningfully above the ~0.006 baseline, the box-jitter
augmentation worked — update the README's honest-negative section with the new number.